# Why the p+p underlying event is flat, and what C_v can do about it

STAR's transverse density (charged, p_T > 0.2 GeV, |η| < 1; PRD 101, 052004) falls from 0.76 to 0.49 between 6 and
40 GeV leading jets. The X-SCAPE chain (PythiaIsrGun → iMATTER → MATTER → iColored) is flat. This notebook shows
the three checks from the tuning page (sections *Why the UE is flat* and *UE trend vs C_v*, 2026-11-24):

1. **Standalone Pythia with ISR off** keeps the falling trend, so the missing trend is not caused by PythiaIsrGun running Pythia without ISR.
2. **Decomposition** (MPI off, then FSR off, then ISR off) of the chain and of Pythia: remnants and MPI fall in both;
   the chain's iMATTER ISR and MATTER FSR rise with jet p_T and cancel the fall.
3. **The 32 hard-tune design points**: the trend against C_v (and α_s, Q₀, and the MPI cross section), and the STAR jets
   above 35 GeV. No C_v on the valley gives a falling trend together with better high-p_T jets.

**Inputs (no new events here).** The UE sums cached by `ue_check.py` (`results/ue_<run>.npz`, git-ignored), the
runs' observables (`$OO_RHIC_PROD/obs`), the hard-tune χ² of the design points (`results/design_chi2.json`) and the
jobs' Pythia logs (for σ_int). All vacuum chain (no strings, no hydro) and, for the X-SCAPE runs, the **old MPI**
(`MultipartonInteractions:pTmin = 5`); the tune XMLs moved to soft MPI (pT0Ref 1.48) afterwards (js-contrib `073085b`).
Missing sums are recomputed from the event files with `ue_check.run_sums`.

| Run | What | Events |
|---|---|---|
| `best`, `best_nompi` | tune point (α_s 0.153, C_v 0.319, Q₀ 1.52), MPI on / off | 900k / 600k |
| `best_nompi_fsroff`, `best_nompi_bare` | MPI off + `matter_on 0`; + `Q0 = 50` (no iMATTER ISR) | 300k each |
| `pythia8_default(_isroff)`, `pythia8_detroit(_isroff)` | standalone Pythia 8.312, `refs/pythia_ref.py` | 600k each; ISR off 300k each |
| `pythia8_default_nompi(_fsroff, _bare)` | Monash, `PartonLevel:MPI / FSR / ISR = off` in turn | 300k each |
| `p000`–`p031` | hard-tune design (maximin LHS) | 480k each |
| `valley_lo`, `ref`, `best`, `valley_hi` | validation runs along the α_s–C_v valley | 300k / 900k / 900k / 300k |

Model results, not vetted physics results; statistical errors only.

In [ ]:
import sys as _sys
_sys.path.insert(0, "..")
import _paths  # noqa: E402,F401  (analysis/OO-RHIC and its test directories on sys.path; shared results/, refs/, data/ in ..)
import glob
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import config as C          # noqa: E402
import make_design as D     # noqa: E402
import observables as O     # noqa: E402
import tune as T            # noqa: E402
import ue_check as U        # noqa: E402

# palette: categorical slots in fixed order (validated reference palette), ink for data
INK, MUTED, GRID = "#1f1f1e", "#6b6a65", "#e4e3de"
S = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUES = LinearSegmentedColormap.from_list("blues", ["#86b6ef", "#3987e5", "#2a78d6", "#1c5cab", "#0d366b"])
ORANGES = LinearSegmentedColormap.from_list("oranges", ["#f7b99b", "#eb6834", "#b4461b", "#7a2e10"])
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 6})

lo, hi = U.LEAD_EDGES[:-1], U.LEAD_EDGES[1:]
mid = 0.5 * (lo + hi)
I13, I40 = list(lo).index(11), list(lo).index(35)
star = U.load_star()["transverse"]
SV = np.asarray(star["value"])
SE = np.hypot(np.asarray(star["stat"]), np.asarray(star["sys"]))
STAR_TREND = SV[I13] / SV[I40]


def ue(run):
    # transverse density and its error per leading-jet bin (ue_check.py definitions)
    npz = C.ROOT / "results" / f"ue_{run}.npz"
    if npz.exists():
        acc = dict(np.load(npz))
    else:
        acc = U.run_sums(run, 16)
        np.savez(npz, **acc)
    return U.densities(acc)["n_transverse"]


def incomplete(ax):
    ax.axvspan(5, 11, color=GRID, alpha=0.7, lw=0, zorder=0)
    ax.text(5.4, 0.03, "incomplete\n(no p̂T < 5 GeV)", transform=ax.get_xaxis_transform(), va="bottom",
            fontsize=7.5, color=MUTED)


def star_points(ax, scale=1.0, label="STAR"):
    ax.errorbar(mid, SV * scale, yerr=SE * scale, xerr=(hi - lo) / 2, fmt="o", color=INK, ms=5,
                lw=1.2, capsize=0, label=label, zorder=5)

print(f"STAR trend UE(11-15)/UE(35-45) = {STAR_TREND:.3f}")

## 1. Standalone Pythia with ISR off keeps the falling trend

`PythiaIsrGun.cc` sets `PartonLevel:ISR = off` with MPI on (iMATTER does the ISR). If the beam depletion by ISR
were what makes Pythia's UE fall, Pythia with ISR off should be flat. It is not: switching ISR off lowers the level by
about 18 % and leaves the slope alone (Monash and STAR's Detroit tune, 10 × 30k events each with ISR off).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.0))
incomplete(ax)
star_points(ax, label="STAR (PRD 101, 052004)")
series = [("best", "X-SCAPE tune (vacuum, old MPI)", S[0], "-"),
          ("pythia8_default", "Pythia 8 Monash", S[1], "-"),
          ("pythia8_default_isroff", "Monash, ISR off", S[1], "--"),
          ("pythia8_detroit", "Pythia 8 Detroit", S[2], "-"),
          ("pythia8_detroit_isroff", "Detroit, ISR off", S[2], "--")]
rows = []
for run, label, c, ls in series:
    v, e = ue(run)
    ax.plot(mid, v, ls, color=c, marker="o", label=label)
    ax.fill_between(mid, v - e, v + e, color=c, alpha=0.15, lw=0)
    rows.append((label, v[I13], v[I40], v[I13] / v[I40]))
ax.set_xlabel("leading-jet $p_T$ [GeV]")
ax.set_ylabel(r"transverse $dN_{ch}/d\eta\,d\phi$  ($p_T$ > 0.2 GeV)")
ax.set_xlim(5, 45)
ax.set_ylim(0.3, 0.86)
ax.legend(ncol=3, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.16))
ax.set_title("Transverse density: ISR off lowers Pythia's level, not its slope", loc="left", fontsize=10.5)
plt.tight_layout()
plt.show()
print(f"{'':34s} 11-15    35-45   trend (STAR {STAR_TREND:.2f})")
for label, a, b, t in rows:
    print(f"{label:34s} {a:.3f}   {b:.3f}   {t:.2f}")

### Number of MPI against p̂_T (parton level)

Pythia rescales the MPI parton densities by what the hard scattering took (x₁ + x₂), whether its own ISR runs or not.
With PythiaIsrGun's settings (ISR and FSR off, `pSet 20`, MPI α_s 0.152, pTmin 5) the same depletion acts on about
0.5 MPI per event instead of about 3.4. Parton level only (`HadronLevel:all = off`), 4000 events per window and setting,
about half a minute.

In [ ]:
import pythia8

WIN = [(5, 7), (9, 11), (14, 18), (23, 30), (30, 40), (40, 55)]
GUN = ["PartonLevel:ISR = off", "PartonLevel:FSR = off", "HardQCD:nQuarkNew = 3",
       "MultipartonInteractions:processLevel = 0", "MultipartonInteractions:nQuarkIn = 3",
       "PDF:pSet = 20", "SigmaProcess:alphaSvalue = 0.152",
       "MultipartonInteractions:alphaSvalue = 0.152", "MultipartonInteractions:pTmin = 5.0"]
CONF = {"Monash, ISR on": [], "Monash, ISR off": ["PartonLevel:ISR = off"], "PythiaIsrGun settings": GUN}
NEV = 4000


def n_mpi(lines, w, seed):
    py = pythia8.Pythia("", False)
    for l in ["Beams:eCM = 200", "HardQCD:all = on", f"PhaseSpace:pTHatMin = {w[0]}",
              f"PhaseSpace:pTHatMax = {w[1]}", "Random:setSeed = on", f"Random:seed = {seed}",
              "HadronLevel:all = off", "Next:numberCount = 0", "Print:quiet = on"] + lines:
        assert py.readString(l), l
    py.init()
    nm = [py.infoPython().nMPI() - 1 for _ in range(NEV) if py.next()]
    return np.mean(nm), np.std(nm) / np.sqrt(len(nm))

nmpi = {c: np.array([n_mpi(l, w, 4242 + 17 * k) for k, w in enumerate(WIN)]) for c, l in CONF.items()}
wmid = np.array([0.5 * (a + b) for a, b in WIN])

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for (c, r), col, ls in zip(nmpi.items(), [S[1], S[1], S[0]], ["-", "--", "-"]):
    axs[0].errorbar(wmid, r[:, 0], yerr=r[:, 1], fmt="o" + ls, color=col, label=c)
    axs[1].plot(wmid, r[:, 0] / r[0, 0], "o" + ls, color=col, label=c)
axs[0].set_ylabel(r"$\langle N_{MPI}\rangle$ besides the hard scattering")
axs[1].set_ylabel("relative to p̂T 5–7 GeV")
for a in axs:
    a.set_xlabel(r"$\hat p_T$ [GeV]")
axs[0].legend(fontsize=8.5)
axs[1].set_title("same depletion in all three, on very different counts", loc="left", fontsize=9.5)
plt.tight_layout()
plt.show()
for c, r in nmpi.items():
    print(f"{c:24s} " + "  ".join(f"{v:.2f}" for v in r[:, 0]))

## 2. Decomposition: what rises and what falls

Each step switches one more piece off; the differences are the pieces.

- **X-SCAPE** at the tune point, `best_nompi`'s XML and seeds: MPI off → FSR off (`<matter_on> 0`) → bare (also `<Q0> 50`, which puts iMATTER's ISR floor above every window). Final partons per event at p̂_T 30–40 GeV: 8.9 / 3.3 / 2.0.
- **Pythia Monash**: `PartonLevel:MPI = off` → `FSR = off` → `ISR = off`.

Caveats: the pieces are not strictly additive (one string hadronisation for everything); FSR off also removes the
showering of ISR emissions, in both models alike; the couplings differ (Monash α_s 0.130 with `pSet 13`, the chain 0.153 with `pSet 20`).

In [ ]:
def diff(a, b):
    (va, ea), (vb, eb) = ue(a), ue(b)
    return va - vb, np.hypot(ea, eb)

parts = {
    "X-SCAPE": {"remnants + bare hard pair": ue("best_nompi_bare"),
                "ISR (iMATTER)": diff("best_nompi_fsroff", "best_nompi_bare"),
                "FSR (MATTER)": diff("best_nompi", "best_nompi_fsroff"),
                "MPI": diff("best", "best_nompi")},
    "Pythia Monash": {"remnants + bare hard pair": ue("pythia8_default_nompi_bare"),
                      "ISR (iMATTER)": diff("pythia8_default_nompi_fsroff", "pythia8_default_nompi_bare"),
                      "FSR (MATTER)": diff("pythia8_default_nompi", "pythia8_default_nompi_fsroff"),
                      "MPI": diff("pythia8_default", "pythia8_default_nompi")}}
titles = {"remnants + bare hard pair": "remnants + bare hard pair", "ISR (iMATTER)": "ISR (chain: iMATTER)",
          "FSR (MATTER)": "FSR (chain: MATTER)", "MPI": "MPI"}

fig, axs = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, comp in zip(axs, titles):
    incomplete(ax)
    for (model, d), col, ls in zip(parts.items(), [S[0], S[1]], ["-", "--"]):
        v, e = d[comp]
        ax.plot(mid, v, ls, color=col, marker="o", label=model)
        ax.fill_between(mid, v - e, v + e, color=col, alpha=0.15, lw=0)
    ax.set_title(titles[comp], loc="left", fontsize=10)
    ax.set_xlabel("leading-jet $p_T$ [GeV]")
    ax.set_xlim(5, 45)
    ax.set_ylim(bottom=0)
axs[0].set_ylabel(r"contribution to $dN_{ch}/d\eta\,d\phi$")
axs[0].legend(fontsize=8.5, loc="lower left")
plt.tight_layout()
plt.show()

print(f"{'':28s}{'':16s} " + "  ".join(f"{a:.0f}-{b:.0f}".rjust(6) for a, b in zip(lo[[0, 3, 5, 7]], hi[[0, 3, 5, 7]])))
for comp in titles:
    for model, d in parts.items():
        v, _ = d[comp]
        print(f"{comp:28s}{model:16s} " + "  ".join(f"{x:6.3f}" for x in v[[0, 3, 5, 7]]))

**Reading.** Remnants fall alike (about −32 % in both). The MPI fall in both too, but the chain's are half of Pythia's.
What differs is the radiation: iMATTER gives almost no transverse activity at low p̂_T (it radiates between Q₀ and
t_max = C_v p̂_T², little room at 6 GeV) and reaches Pythia's ISR only at 35–45 GeV; MATTER's FSR (from C_v p_T²)
rises three times as much as Pythia's. Both rising pieces scale with C_v, which is why the next section scans it.

## 3. The trend against C_v on the hard-tune design

Trend = UE(11–15 GeV) / UE(35–45 GeV), STAR 1.34; jets = model / STAR at |η| < 0.5 in the two bins 37–52 GeV.

**A complication in the old-MPI runs.** With `pTmin = 5` Pythia lowers pT0 in 10 % steps until σ_int > 1.1 σ_ND, so
σ_int lands between about 31 and 53 mb from point to point, independent of C_v (read here from each run's first log).
More MPI make the UE higher and steeper. A linear fit in (C_v, α_s, Q₀, σ_int) over the 32 points separates the
two; the valley runs serve as a check. Soft MPI with a fixed pT0Ref (the new XMLs) has no such step.

In [ ]:
design = D.load()
data = O.load_data()
m0 = T.fit_mask("star_jets_eta0", data)
e0 = np.asarray(data["star_jets_eta0"]["edges"])[m0]
d0 = np.asarray(data["star_jets_eta0"]["value"])[m0]
JET_HI = e0[:, 0] >= 37
chi2_hard = {d["point"]: d["chi2"] for d in json.load(open(C.ROOT / "results" / "design_chi2.json"))}
VALLEY = ["valley_lo", "ref", "best", "valley_hi"]


def sig_int(run):
    log = sorted(glob.glob(str(C.point_dir(run) / "*.log")))[0]
    for line in open(log):
        if "accepted" in line and "sigmaInteraction" in line:
            return float(re.search(r"=\s*([\d.]+) mb", line).group(1))
    return np.nan


rows = []
for r in list(design["points"]) + VALLEY:
    v, e = ue(r)
    trend = v[I13] / v[I40]
    jv, _ = O.combine(dict(np.load(C.obs_path(r))), "star_jets_eta0", data)
    p = (design["points"].get(r) or design["extra"][r])["params"]
    rows.append(dict(run=r, a=p["alpha_s"], cv=p["Cv"], q0=p["Q0"], sig=sig_int(r), trend=trend,
                     trend_e=trend * np.hypot(e[I13] / v[I13], e[I40] / v[I40]), l13=v[I13], l40=v[I40],
                     jet=float((jv[m0] / d0)[JET_HI].mean()), chi2=chi2_hard.get(r, np.nan)))
tab = {k: np.array([r[k] for r in rows]) for k in rows[0]}
isd = np.array([r.startswith("p") for r in tab["run"]])
isv = ~isd

# linear fits over the design points
X = np.column_stack([np.ones(isd.sum()), tab["cv"][isd], tab["q0"][isd], tab["a"][isd], tab["sig"][isd]])
NAMES = ["const", "C_v", "Q0", "alpha_s", "sigma_int/mb"]
fits = {}
for key in ("trend", "l13", "l40", "jet"):
    y = tab[key][isd]
    c, *_ = np.linalg.lstsq(X, y, rcond=None)
    rms = np.sqrt(((y - X @ c) ** 2).sum() / (len(y) - X.shape[1]))
    fits[key] = (c, np.linalg.inv(X.T @ X) * rms ** 2, rms)
    print(f"{key:6s} " + "  ".join(f"{n} {v:+.3f}({np.sqrt(fits[key][1][i, i]):.3f})" for i, (n, v) in enumerate(zip(NAMES, c)))
          + f"   rms {rms:.3f}")

SIG0, Q00 = 38.7, 1.52                      # the best point's sigma_int, the tune's Q0
valley_as = lambda cv: 0.149 + 0.019 * (cv - 0.16)


def along_valley(key, cv):
    c, cov, _ = fits[key]
    x = np.column_stack([np.ones_like(cv), cv, np.full_like(cv, Q00), valley_as(cv), np.full_like(cv, SIG0)])
    return x @ c, np.sqrt(np.einsum("ij,jk,ik->i", x, cov, x))


def to_sig0(key):
    # the runs moved to sigma_int = 38.7 mb with the fit's sigma_int slope
    return tab[key] - fits[key][0][4] * (tab["sig"] - SIG0)

print("\nvalley runs:")
for i in np.where(isv)[0]:
    print(f"  {tab['run'][i]:9s} alpha_s {tab['a'][i]:.3f} C_v {tab['cv'][i]:.2f} Q0 {tab['q0'][i]:.2f} "
          f"sigma_int {tab['sig'][i]:.1f} mb: trend {tab['trend'][i]:.2f}+-{tab['trend_e'][i]:.2f} "
          f"(at {SIG0} mb: {to_sig0('trend')[i]:.2f}), jets {tab['jet'][i]:.2f}")

In [ ]:
cvg = np.linspace(0.16, 0.50, 60)
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
norm = plt.Normalize(0.11, 0.20)
for ax, y, title in [(axs[0], tab["trend"], "as run (σ_int 31–53 mb)"),
                     (axs[1], to_sig0("trend"), f"moved to σ_int = {SIG0} mb (fit slope)")]:
    ax.axhline(STAR_TREND, color=INK, ls=":", lw=1.2)
    ax.text(0.495, STAR_TREND + 0.012, f"STAR {STAR_TREND:.2f}", ha="right", fontsize=8.5, color=INK)
    ax.axhline(1.0, color=MUTED, lw=0.8)
    sc = ax.scatter(tab["cv"][isd], y[isd], c=tab["a"][isd], cmap=BLUES, norm=norm, s=46,
                    edgecolor="white", linewidth=1.2, zorder=3)
    ax.scatter(tab["cv"][isv], y[isv], marker="D", s=58, facecolor=S[1], edgecolor="white",
               linewidth=1.2, zorder=4, label="valley runs")
    for i in np.where(isv)[0]:
        ax.annotate(tab["run"][i], (tab["cv"][i], y[i]), xytext=(4, -13), textcoords="offset points",
                    fontsize=7.5, color=INK)
    ax.set_xlim(0.14, 0.55)
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel("$C_v$")
m, s = along_valley("trend", cvg)
axs[1].plot(cvg, m, color=S[1], lw=2, label="fit along the valley")
axs[1].fill_between(cvg, m - s, m + s, color=S[1], alpha=0.18, lw=0)
for name, off in (("p026", (6, 4)), ("p029", (10, -1)), ("p001", (6, 2)), ("p025", (6, 2))):
    i = list(tab["run"]).index(name)
    axs[1].annotate(f"{name} (hard χ² {tab['chi2'][i]:.0f})", (tab["cv"][i], to_sig0("trend")[i]), xytext=off,
                    textcoords="offset points", fontsize=7.5, color=MUTED)
axs[0].set_ylabel("UE trend  UE(11–15) / UE(35–45 GeV)")
axs[1].legend(fontsize=8.5, loc="lower left")
cb = fig.colorbar(sc, ax=axs, pad=0.01, fraction=0.03)
cb.set_label(r"$\alpha_s(M_Z)$ (design points)")
fig.suptitle("Lower C_v steepens the fall, but not to STAR's within the design range", x=0.06, ha="left", fontsize=11)
plt.show()

for cv in (0.16, 0.25, 0.32, 0.40, 0.50):
    out = {k: along_valley(k, np.array([cv])) for k in ("trend", "l13", "l40", "jet")}
    print(f"valley C_v {cv:.2f} (alpha_s {valley_as(cv):.4f}): trend {out['trend'][0][0]:.2f}+-{out['trend'][1][0]:.2f}"
          f"  UE 11-15 {out['l13'][0][0]:.3f}  35-45 {out['l40'][0][0]:.3f}  jets 37-52 {out['jet'][0][0]:.2f}")

## 4. High-p_T jets: set by α_s, not by C_v

The jet ratio above 35 GeV moves with α_s (about +9 per unit) and not with C_v (+0.01 ± 0.19). Along the valley it
stays at 0.70–0.76. The design points with jets near 1 sit at α_s 0.18–0.20, where the hard-tune χ² is 150–370 / 35.
Right: the two goals together. STAR's corner (jets 1, trend 1.34) is empty; the points that come closest
in either direction are far off the valley.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax = axs[0]
ax.axvspan(0.147, 0.159, color=GRID, alpha=0.8, lw=0, zorder=0)
ax.text(0.148, 0.04, "α_s posterior 95 %", transform=ax.get_xaxis_transform(), fontsize=8, color=MUTED)
ax.axhline(1.0, color=INK, ls=":", lw=1.2)
sc = ax.scatter(tab["a"][isd], tab["jet"][isd], c=tab["cv"][isd], cmap=ORANGES, vmin=0.16, vmax=0.5, s=46,
                edgecolor="white", linewidth=1.2, zorder=3)
ax.scatter(tab["a"][isv], tab["jet"][isv], marker="D", s=58, facecolor=S[0], edgecolor="white", linewidth=1.2,
           zorder=4, label="valley runs")
cb = fig.colorbar(sc, ax=ax, pad=0.01, fraction=0.05)
cb.set_label("$C_v$")
ax.set_xlabel(r"$\alpha_s(M_Z)$")
ax.set_ylabel("STAR jets, model / data, 37–52 GeV")
ax.legend(fontsize=8.5, loc="upper left")
ax.set_title("jets above 35 GeV follow α_s", loc="left", fontsize=10)

ax = axs[1]
good = isd & (tab["chi2"] < 45)
bad = isd & ~good
yt = to_sig0("trend")
ax.scatter(tab["jet"][bad], yt[bad], s=40, facecolor="none", edgecolor=MUTED, linewidth=1.2,
           label="design, hard χ² ≥ 45 / 35")
ax.scatter(tab["jet"][good], yt[good], s=50, facecolor=S[0], edgecolor="white", linewidth=1.2, zorder=3,
           label="design, hard χ² < 45 / 35")
ax.scatter(tab["jet"][isv], yt[isv], marker="D", s=58, facecolor=S[1], edgecolor="white", linewidth=1.2,
           zorder=4, label="valley runs")
ax.plot([1.0], [STAR_TREND], marker="*", ms=16, color=INK, ls="none", label="STAR (jets 1, trend 1.34)")
LAB = {"valley_lo": (8, 3), "ref": (-28, 4), "best": (-30, 4), "valley_hi": (-52, -4),
       "p026": (6, 3), "p029": (6, -9), "p001": (6, 3), "p025": (6, -3), "p028": (6, -3), "p031": (-6, 8)}
for name, off in LAB.items():
    i = list(tab["run"]).index(name)
    ax.annotate(name, (tab["jet"][i], yt[i]), xytext=off, textcoords="offset points",
                fontsize=7.5, color=INK if isv[i] else MUTED)
ax.set_xlabel("STAR jets, model / data, 37–52 GeV")
ax.set_ylabel(f"UE trend at σ_int = {SIG0} mb")
ax.legend(fontsize=8, loc="lower right")
ax.set_title("both goals together", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

print("design points closest to the valley in the hard fit (chi2 < 45 / 35):")
for i in np.argsort(tab["chi2"]):
    if good[i]:
        print(f"  {tab['run'][i]}  alpha_s {tab['a'][i]:.3f} C_v {tab['cv'][i]:.2f} Q0 {tab['q0'][i]:.2f}  chi2 {tab['chi2'][i]:.1f}"
              f"  trend {tab['trend'][i]:.2f} (at {SIG0} mb {yt[i]:.2f})  jets {tab['jet'][i]:.2f}")

## 5. Level along the valley

The rising radiation shows most at 35–45 GeV. Old MPI, vacuum chain: the level has to be compared with the soft-MPI
full chain on the tuning page (*G2, soft MPI*), which adds MPI and strings + hydro on top.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for key, idx, col, lab in [("l13", I13, S[0], "11–15 GeV"), ("l40", I40, S[1], "35–45 GeV")]:
    m, s = along_valley(key, cvg)
    ax.plot(cvg, m, color=col, label=f"fit along the valley, {lab}")
    ax.fill_between(cvg, m - s, m + s, color=col, alpha=0.18, lw=0)
    ax.scatter(tab["cv"][isv], to_sig0(key)[isv], marker="D", s=50, facecolor=col, edgecolor="white", zorder=3)
    ax.axhline(SV[idx], color=col, ls=":", lw=1.2)
    ax.text(0.497, SV[idx] + 0.008, f"STAR {lab}", ha="right", fontsize=8, color=INK)
ax.set_xlabel("$C_v$")
ax.set_ylabel(r"transverse $dN_{ch}/d\eta\,d\phi$ (vacuum, old MPI)")
ax.legend(fontsize=8.5, loc="upper left")
ax.set_title(f"level at σ_int = {SIG0} mb; diamonds: valley runs moved there", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Summary

- The flat UE is not a PythiaIsrGun problem: Pythia without ISR still falls, and its MPI depletion comes from the hard x₁ + x₂.
- In the chain, remnants and MPI fall as in Pythia; iMATTER ISR and MATTER FSR rise with jet p_T and cancel the fall. The MPI part is about half of Pythia's.
- Lower C_v steepens the fall (fit: trend ≈ 1.17 at C_v 0.16 against 1.01 at 0.32, at a common σ_int); STAR's 1.34 is not reached inside the design range.
- Jets above 35 GeV depend on α_s only (0.70–0.76 along the valley). No C_v on the valley gives a falling UE together with better high-p_T jets.
- Changing C_v is not p+p-only: it is also MATTER's vir_factor in the O+O medium, and Q₀/√C_v reaches the lowest window (5 GeV) at C_v 0.16 with Q₀ 2.

Model results, not vetted physics results.